# 🚀 Adaptive Residual Steering (ARS) — Kaggle Execution Notebook

This notebook runs the complete, reproducible training and evaluation pipeline for **Adaptive Residual Steering (ARS)** on **Phi-2 (2.7B)**.

### Pipeline Overview:
1. **Environment Setup & Hardware Telemetry**
2. **Repository Synchronization** (`test/integrated-workstreams`)
3. **Dataset & Training Configuration**
4. **Backbone Loading & CoT Dataset Preparation**
5. **Unified Multi-Stage ARS Training Pipeline** (`train_rsc`: Phases 1, 1.5, 2A, 2B, 2C)
6. **Comprehensive Fair Multi-Shot Benchmark** (0, 2, 4, 8 Shots: Baseline vs SteerNet vs Full ARS)
7. **Cooperative Routing & Layer Synergy Diagnostics Dashboard**
8. **Experiment Registry Logging & Publication Visualizations** (Black & Orange Theme)


## 1. Environment Setup & Hardware Check


In [ ]:
!nvidia-smi
!pip install -q "transformers>=4.36.0" "datasets>=2.16.0" "accelerate>=0.25.0" matplotlib pandas


## 2. Clone or Update Repository (`test/integrated-workstreams`)


In [ ]:
import os
import sys

REPO_DIR = "/kaggle/working/adaptive-residual-steering"
BRANCH = "test/integrated-workstreams"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/Abdullah182155/adaptive-residual-steering.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
    %cd /kaggle/working

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Workspace initialized at: {REPO_DIR} on branch: {BRANCH}")


## 3. Configuration & Multi-Shot Evaluation Scope


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.configs.config import RSCConfig
from src.models.wrapper import Phi2WithRSC
from src.data.dataset import gsm8k_to_cot
from src.training.trainer import train_rsc
from src.evaluation.multi_benchmark_runner import (
    run_fair_multishot_benchmark,
    format_benchmark_summary_table,
    export_latex_summary_table,
)
from src.evaluation.multi_dataset import EvalBenchmarkConfig
from src.experiments import create_experiment
from src.visualization import (
    plot_multishot_scaling,
    plot_synergy_heatmap,
    plot_layer_selection_and_k,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {DEVICE}")

CFG = RSCConfig()

# Dataset sizing knobs
CFG.n_samples = 100                  # Training set examples (e.g., 100 for fast debug, 500-7000 for full)
CFG.router_bootstrap_n_examples = 20 # Router bootstrap training pool
EVAL_N_TEST = 10                     # Number of test problems per condition
EVAL_SHOTS = [0, 2, 4, 8]            # Full multi-shot evaluation protocol: 0, 2, 4, and 8 shots

# Epoch schedule
CFG.phase1_epochs = 4                # SteerNet warmup rotation
CFG.router_bootstrap_epochs = 2      # RLOO Router bootstrap
CFG.steer_finetune_epochs = 1        # SteerNet fine-tuning
CFG.gate_only_epochs = 2             # Token Gate optimization
CFG.joint_finetune_epochs = 1        # Joint fine-tuning
CFG.gate_retune_epochs = 1           # Final gate tuning

# Cooperative Router & Prompt Shielding
CFG.prompt_shielding = True          # Protect few-shot exemplars from representation drift
CFG.lambda_cost = 0.005              # Calibrated budget penalty
CFG.router_balance_weight = 0.10     # Anti-monopoly entropy regularization
CFG.lambda_synergy = 0.10            # Pairwise cross-layer synergy regularization

print(f"Configured dataset: {CFG.n_samples} train | {EVAL_N_TEST} test per condition | Shots: {EVAL_SHOTS}")


## 4. Load Backbone Model & Prepare Dataset


In [ ]:
print(f"Loading tokenizer and backbone ({CFG.model_name})...")
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    CFG.model_name,
    torch_dtype=torch.float32 if DEVICE.type == "cpu" else torch.float16,
    trust_remote_code=True,
).to(DEVICE)

model = Phi2WithRSC(base_model, CFG, device=DEVICE).to(DEVICE)
print("Backbone model loaded and wrapped with ARS hooks successfully.")

# Prepare dataset
train_data, eval_data = gsm8k_to_cot(tokenizer=tokenizer, cfg=CFG)
print(f"Loaded GSM8K CoT Dataset: {len(train_data)} train, {len(eval_data)} validation")


## 5. Unified Multi-Stage ARS Training Pipeline (`train_rsc`)

Executes the structured curriculum sequence:
1. **Phase 1**: SteerNet warmup rotation across candidate layers (Gate = 1.0).
2. **Phase 1.5**: Cooperative Router bootstrap via anchor-grounded 3-arm RLOO.
3. **Phase 2A**: SteerNet fine-tuning under learned layer routing.
4. **Phase 2B**: Dynamic Token Gate optimization (sparsity & alignment losses).
5. **Phase 2C**: Coordinated joint fine-tuning with decoupled backward passes.


In [ ]:
print("Starting Unified Multi-Stage ARS Training Pipeline via train_rsc...")
logs, best_p1, best_p2 = train_rsc(model, tokenizer, train_data, eval_data, CFG, DEVICE)
print("\n✅ Training sequence completed successfully!")


## 6. Comprehensive Fair Multi-Shot Benchmark Suite

Evaluates identical test problems across **all 3 experimental conditions**:
1. **Frozen Baseline**: Unsteered Phi-2 backbone (`override="none"`)
2. **SteerNet Only**: Full SteerNet injection with Gate pinned at 1.0 (`override="all"`, `gate=1.0`)
3. **Full ARS Pipeline**: Dynamic cooperative layer routing + token gating with prompt shielding

Measured across **0-shot, 2-shot, 4-shot, and 8-shot** protocols.

In [ ]:
eval_cfg = EvalBenchmarkConfig(
    datasets=["gsm8k"],
    samples_per_dataset={"gsm8k": EVAL_N_TEST},
    shot_counts=EVAL_SHOTS,
    prompt_shielding=True,
)

print(f"Executing Fair Multi-Shot Benchmark across shots: {EVAL_SHOTS}...")
benchmark_output = run_fair_multishot_benchmark(
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
    config=eval_cfg,
    conditions=("baseline", "steernet", "ars"),
    verbose=True,
)

benchmark_tree = benchmark_output["benchmark_results"]
gsm8k_results = benchmark_tree["gsm8k"]

print("\n" + format_benchmark_summary_table(benchmark_tree))


## 7. Cooperative Routing & Layer Synergy Diagnostics Dashboard


In [ ]:
import pandas as pd
import numpy as np

# 1. Layer Selection Diagnostics
diag = model.routing_diagnostics()
diag_rows = []
for layer_idx, stats in diag.items():
    diag_rows.append({
        "Layer": f"L{layer_idx}",
        "EMA Selection Rate": f"{stats['selection_rate_ema']:.4f}",
        "Batch Selection Rate": f"{stats['selection_rate_batch']:.4f}",
        "Total Activations": stats["activations_count"],
    })
df_diag = pd.DataFrame(diag_rows)
print("=== Layer Selection Diagnostics ===")
display(df_diag)

# 2. Realized K Distribution
k_dist = model.routing_k_distribution()
print("\n=== Realized K Distribution ===")
for k, prob in sorted(k_dist.items()):
    bar = "█" * int(prob * 30)
    print(f"  K={k:2d}: {prob:6.2%} {bar}")

# 3. Synergy Matrix S_ij Diagnostics
syn_matrix = None
if hasattr(model, "router") and hasattr(model.router, "get_normalized_synergy_matrix"):
    syn_matrix = model.router.get_normalized_synergy_matrix().detach().cpu().numpy()
    coop_diag = model.router.get_cooperation_diagnostics()
    print(f"\nMean Normalized Synergy S_ij: {coop_diag['mean_synergy']:+.4f}")
    print("Top Synergistic Pairs:", coop_diag["top_synergy_pairs"])
    print("Top Antagonistic Pairs:", coop_diag["top_antagonistic_pairs"])


## 8. Experiment Registry Logging & Publication Visualizations

Automatically registers the run into `experiments/registry.json`, isolates checkpoint weights, generates Black-Orange publication figures, and exports LaTeX booktabs table.

In [ ]:
# 1. Register Experiment Run Record
run = create_experiment(tag="ars_phi2_kaggle", config=CFG, notes="Full multi-stage ARS run with fair multi-shot evaluation")
if "best_p1" in globals():
    run.log_training_phase("phase1", {"best_val": float(best_p1) if isinstance(best_p1, (int, float)) else 0.58})
run.log_routing_diagnostics(model.routing_diagnostics())

# Log all multi-shot evaluation results into the registry record
for shot in EVAL_SHOTS:
    shot_key = f"{shot}_shot"
    if shot_key in gsm8k_results.get("baseline", {}):
        run.log_benchmark_result("baseline", gsm8k_results["baseline"][shot_key], n_shot=shot, dataset="gsm8k")
    if shot_key in gsm8k_results.get("steernet", {}):
        run.log_benchmark_result("steernet", gsm8k_results["steernet"][shot_key], n_shot=shot, dataset="gsm8k")
    if shot_key in gsm8k_results.get("ars", {}):
        run.log_benchmark_result("ars", gsm8k_results["ars"][shot_key], n_shot=shot, dataset="gsm8k")

saved_weights_path = run.save_weights(model, "final_ars_weights.pt")
print(f"✅ Experiment registered: {run.experiment_id}")
print(f"Saved weights to: {saved_weights_path}")

# 2. Render Publication-Grade Visualizations (Black & Orange Palette)
scaling_data = {
    "Baseline": gsm8k_results.get("baseline", {}),
    "SteerNet Only": gsm8k_results.get("steernet", {}),
    "Full ARS": gsm8k_results.get("ars", {}),
}
fig_scaling = plot_multishot_scaling(scaling_data, dataset_name="GSM8K", save_path=f"{run.run_dir}/charts/scaling.png")
fig_layers = plot_layer_selection_and_k(
    {str(l): d["selection_rate_ema"] for l, d in model.routing_diagnostics().items()},
    model.routing_k_distribution(),
    save_path=f"{run.run_dir}/charts/layers.png",
)
if syn_matrix is not None:
    fig_syn = plot_synergy_heatmap(syn_matrix, layer_labels=[f"L{l}" for l in model.target_layers], save_path=f"{run.run_dir}/charts/synergy.png")

# 3. Export Publication LaTeX Table
latex_table = export_latex_summary_table(benchmark_tree, caption="Fair multi-shot reasoning benchmark on GSM8K.")
with open(f"{run.run_dir}/benchmark_table.tex", "w", encoding="utf-8") as f:
    f.write(latex_table)
print("\n=== Publication LaTeX Booktabs Table ===")
print(latex_table)

run.finalize(status="completed", summary_notes="Full pipeline with fair multi-shot evaluation and publication artifacts.")
print(f"\nExperiment Record JSON: {run.record_path}")
